# Evaluation setup: which published models exist, and on which peptides we test them

This module tests **pretrained, published** half-life regressors on our own measurements.
No model is trained or refitted here, so every number describes a published artifact as
distributed by its authors.

This notebook fixes the three things the per-model notebooks depend on: the published
models that can actually be obtained, the evaluation cohort, and the measurement interval
estimated around each measured point.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/model_benchmark").is_dir())
sys.path.insert(0, str(ROOT / "src"))

%matplotlib inline
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

## Which published half-life models can be evaluated

A published accuracy figure can only be checked when the trained parameters are
distributed. The registry separates artifacts we could load from models that exist only
as a web service or only as a paper.

In [2]:
from model_benchmark.predictors import registry, UNAVAILABLE_MODELS, VARIANTS

available = pd.DataFrame(
    [
        {
            "predictor": key,
            "estimator": VARIANTS[key].estimator,
            "representation": VARIANTS[key].backbone,
            "input": VARIANTS[key].input_kind,
            "target": VARIANTS[key].target,
        }
        for key in registry()
    ]
)
display(available.style.set_caption("Published artifacts loaded locally"))

unavailable = pd.DataFrame(UNAVAILABLE_MODELS).T.reset_index(names="model")
display(unavailable.style.set_caption("Published half-life models that could not be evaluated, with the reason"))

## A published model without weights can still be measured when its authors run a
## service and permit automated use. The evidence for each decision is recorded.
from model_benchmark.predictors import BLOCKED_SERVICES, WEB_SERVICES

print("Queried over HTTP:", ", ".join(sorted(WEB_SERVICES)))
display(pd.DataFrame(BLOCKED_SERVICES).T.reset_index(names="service")
        .style.set_caption("Services that were deliberately not queried, with what was observed"))

,predictor,estimator,representation,input,target
0,peptiverse_transformer_esm_log,transformer,esm,sequence,log1p
1,peptiverse_transformer_esm_raw,transformer,esm,sequence,raw
2,peptiverse_xgb_esm_log,xgb,esm,sequence,log1p
3,peptiverse_xgb_esm_raw,xgb,esm,sequence,raw
4,peptiverse_transformer_peptideclm,transformer,peptideclm,smiles,raw
5,peptiverse_xgb_peptideclm,xgb,peptideclm,smiles,raw
6,peptiverse_transformer_chemberta,transformer,chemberta,smiles,raw
7,peptiverse_xgb_chemberta,xgb,chemberta,smiles,raw


,model,publication,target,access,reason
0,hlp,"Sharma et al. 2014, BMC Bioinformatics (HLP)",half-life in an intestine-like environment,web server at webs.iiitd.edu.in/raghava/hlp,"The operator blocks automated submission (HTTP 403) and its robots.txt disallows /cgibin/ for all agents; the matrix is also intestinal, not serum"
1,tan_2024,"Tan et al. 2024, Briefings in Bioinformatics","half-life across species and organs, with enzymatic cleavage features",no code or weight release located,Trained artifact not published; the reported R^2 cannot be reproduced
2,pepmsnd,"Hu et al. 2025, Digital Discovery (PepMSND)","blood stability as a threshold label, not a half-life regression",web server at model.highslab.com/pepmsnd,"The host did not respond over HTTP or HTTPS during this study; the target is also a threshold label, and its 635 training records are a source of our dataset"
3,cavaco_2021,"Cavaco et al. 2021, Clinical and Translational Science",serum half-life from physicochemical properties,publisher page returned HTTP 403 during this study,"Published coefficients not retrieved, so the fitted model could not be reconstructed"


Queried over HTTP: plifepred_svm, plifepred_svm_motif


,service,publication,endpoint,observed,robots,decision,note
0,hlp,"Sharma et al. 2014, BMC Bioinformatics (HLP)",https://webs.iiitd.edu.in/cgibin/hlp/pep_both.pl,HTTP 403: Submissions from external sources or automated bots are blocked,disallowed for all agents: /cgibin/,not queried; the operator prohibits automated submission,"predicts half-life in an intestine-like environment, not serum"
1,pepmsnd,"Hu et al. 2025, Digital Discovery (PepMSND)",http://model.highslab.com/pepmsnd,connection failed over HTTP and HTTPS (no response),not retrieved; host unreachable,not queried; service unreachable,"classification target, so not a half-life regression benchmark"


## Evaluation cohort

The cohort rule is the contract rule: an exact half-life relation with a positive point
estimate, restricted to the canonical amino-acid alphabet and to peptides shorter than
50 residues. Repeated measurements of one peptide are collapsed to their geometric mean,
because a sequence-only predictor cannot distinguish repeated conditions.

In [3]:
from model_benchmark.config import COHORTS, PRIMARY_COHORT
from model_benchmark.dataset import cohort_summary, load_cohort

summaries = []
for name in COHORTS:
    cohort = load_cohort(name)
    row = cohort_summary(cohort)
    row.insert(0, "cohort", name)
    row.insert(1, "description", COHORTS[name]["description"])
    summaries.append(row)
display(pd.concat(summaries, ignore_index=True).style.set_caption("Available cohorts"))

cohort = load_cohort(PRIMARY_COHORT)
print(f"Primary cohort: {PRIMARY_COHORT}, {len(cohort)} peptides")
display(
    cohort[["sequence", "length", "half_life_seconds", "duration_class", "record_count", "modification_free", "sources"]]
    .head(12)
    .style.set_caption("First peptides of the primary cohort")
)

,cohort,description,peptides,modification_free,records,median_length,min_half_life_s,median_half_life_s,max_half_life_s,log10_range,median_replicate_spread_log10,class_short,class_medium,class_long
0,human_serum_in_vitro,"Human serum, in vitro: the condition published serum predictors target",128,54,153,15.000000,120.000000,4290.000000,421200.000000,3.545307,0.000000,38,55,35
1,human_serum_any,"Human serum, any measurement context",142,62,182,15.000000,120.000000,4026.000000,421200.000000,3.545307,0.000000,47,57,38
2,human_serum_plasma_in_vitro,"Human serum and plasma, in vitro",252,119,325,14.000000,52.560022,3600.000000,582012.000000,4.044276,0.000000,99,87,66
3,blood_in_vitro_any_species,"Serum and plasma of any species, in vitro: largest proteolysis cohort",391,168,563,15.000000,36.000000,4080.000000,843120.000000,4.369587,0.055990,151,132,108


Primary cohort: human_serum_in_vitro, 128 peptides


,sequence,length,half_life_seconds,duration_class,record_count,modification_free,sources
0,RLYE,4,33840.000000,long,1,False,peplife2
1,ALAGIGILTV,10,86400.000000,long,1,False,peplife2
2,RFGRFLRKIRRFRRKDVY,18,21168.000000,medium,1,True,pepmsnd
3,GAARYAGWGAYRAAG,15,3300.000000,medium,1,False,cavaco
4,ELAGIGILTV,10,86400.000000,long,1,False,peplife2
5,DLAGIGILTV,10,86400.000000,long,1,False,peplife2
6,GAARAAGSGAARAAG,15,5220.000000,medium,1,True,pepmsnd
7,EAAGIGILTV,10,2700.000000,medium,1,True,peplife2
8,ELAGIGILTV,10,72000.000000,long,1,False,peplife2
9,RLYE,4,4380.000000,medium,1,True,peplife2


## Measurement interval around a measured point

The logarithmic relative-width estimator of
[`0_03_estimation_interval_analysis`](../../half_time/0_03_estimation_interval_analysis.ipynb)
turns a measured half-life into lower and upper limits. It is used here as the tolerance
band that a prediction should fall into.

**Its domain is narrow.** It is fitted on the 16 published Cavaco intervals in 50% human
serum, so predictions for peptides outside that half-life range are scored by
extrapolation. Every interval hit rate is therefore reported together with the share of
extrapolated peptides.

In [4]:
from model_benchmark.metrics import measurement_interval_metrics, measurement_interval_model

interval_model, reported = measurement_interval_model()
print(f"Fitted on {len(reported)} reported intervals from: {', '.join(sorted(reported.source.unique()))}")
print(f"Reference t0: {interval_model.reference_seconds / 60:.1f} min")
print(f"Fitted domain: {interval_model.domain_seconds[0] / 60:.1f}-{interval_model.domain_seconds[1] / 60:.1f} min")
print("Coefficients [intercept, slope] per interval side:")
print(np.asarray(interval_model.coefficients))

points = cohort.half_life_seconds.to_numpy(dtype=float)
lower, upper = interval_model.predict(points, allow_extrapolation=True)
band = pd.DataFrame(
    {
        "half_life_seconds": points,
        "lower_seconds": lower,
        "upper_seconds": upper,
        "width_log10": np.log10(upper) - np.log10(lower),
        "width_factor": (upper / lower),
    }
)
display(band.describe().loc[["min", "50%", "max"]].style.set_caption("Estimated interval width over the primary cohort"))

outside = (points < interval_model.domain_seconds[0]) | (points > interval_model.domain_seconds[1])
print(f"Peptides outside the fitted domain: {outside.sum()} of {len(points)} ({100 * outside.mean():.0f}%)")

Fitted on 16 reported intervals from: cavaco
Reference t0: 42.5 min
Fitted domain: 8.0-322.0 min
Coefficients [intercept, slope] per interval side:
[[0.46407832 0.48502729]
 [0.03279802 0.09393112]]


,half_life_seconds,lower_seconds,upper_seconds,width_log10,width_factor
min,120.000000,83.401786,146.260953,0.243953,1.753691
50%,4290.000000,2651.591880,7316.552506,0.440797,2.759289
max,421200.000000,223976.995180,1105223.678239,0.693247,4.934541


Peptides outside the fitted domain: 54 of 128 (42%)


## Overlap with published training data

The PeptiVerse repository distributes its half-life training table as SMILES, so the
overlap with our cohort is determined exactly by comparing canonical structures. Peptides
in that overlap measure the model's training fit, not its transfer, and are reported
separately in every per-model notebook.

In [5]:
from model_benchmark.leakage import peptiverse_overlap

matched, overlap_summary = peptiverse_overlap(cohort)
display(pd.DataFrame([overlap_summary]).style.set_caption("Overlap between the primary cohort and the published training set"))
if not matched.empty:
    display(
        matched[["sequence", "log10_half_life", "training_log10_seconds", "cohort_minus_training_log10", "sources"]]
        .sort_values("cohort_minus_training_log10", key=abs, ascending=False)
        .head(12)
        .style.set_caption("Overlapping peptides: our curated value against the published training target")
    )

,cohort_peptides,training_rows,training_structures,overlapping_peptides,overlap_share,median_target_difference_log10
0,128,275,244,26,0.203125,0.308860


,sequence,log10_half_life,training_log10_seconds,cohort_minus_training_log10,sources
88,RRWQWR,4.936514,3.498311,1.438203,pepmsnd
44,RRWWRF,4.936514,3.498311,1.438203,pepmsnd
24,APGPR,2.459392,3.866287,-1.406895,pepmsnd
78,APGPR,2.477121,3.866287,-1.389166,peplife2;plifepred
41,YGGFL,4.276462,3.335877,0.940585,peplife2
13,RRWQWRMKKLG,3.819544,3.255273,0.564271,peplife2
110,RRWQWRMKKLG,3.818754,3.255273,0.563481,pepmsnd
33,HSQGTFTSDYSKYLDSRRAQDFVQWLMNT,2.778151,3.222196,-0.444045,plifepred
74,HSQGTFTSDYSKYLDSRRAQDFVQWLMNT,2.786751,3.222196,-0.435445,pepmsnd
87,YGGFL,3.751279,3.335877,0.415402,peplife2


## Measured single-substitution pairs

The second analysis question needs peptide pairs that differ in exactly one residue and
were measured under the same condition. These pairs are the only empirical check of
whether a predicted change has the right direction.

In [6]:
from model_benchmark.sensitivity import empirical_substitution_pairs

records = load_cohort(PRIMARY_COHORT, aggregate=False)
pairs = empirical_substitution_pairs(records)
print(f"Measured single-substitution pairs in the primary cohort: {len(pairs)}")
if not pairs.empty:
    display(
        pairs[["parent", "variant", "position", "parent_residue", "variant_residue", "measured_delta_log10"]]
        .sort_values("measured_delta_log10", key=abs, ascending=False)
        .style.set_caption("Measured effect of one substitution, in log10 seconds")
    )

Measured single-substitution pairs in the primary cohort: 12


,parent,variant,position,parent_residue,variant_residue,measured_delta_log10
5,EAAGIGILTV,ELAGIGILTV,1,A,L,1.083852
8,GAAQAYGSGYAQAAG,GAAQAYGWGYAQAAG,7,S,W,0.512147
10,GSSQSSGSGSSQSSG,GSSQSSGWGSSQSSG,7,S,W,-0.481106
11,RF,RK,1,F,K,0.397940
9,GAARAAGSGAARAAG,GAARAAGWGAARAAG,7,S,W,-0.383217
6,ELAGIGILTV,GLAGIGILTV,0,E,G,0.288164
3,DLAGIGILTV,ELAGIGILTV,0,D,E,-0.270783
1,ALAGIGILTV,ELAGIGILTV,0,A,E,-0.251889
7,GAAQAAGSGAAQAAG,GAAQAAGWGAAQAAG,7,S,W,0.145481
2,ALAGIGILTV,GLAGIGILTV,0,A,G,0.036275
